# **Shopper Spectrum: Customer Segmentation & Product Recommendations** -


##### **Project Type**    - Unsupervised Learning (Clustering) + Recommendation System
##### **Contribution**    - Individual
##### **Team Member 1 -** Your Name


# **Project Summary -**

This project analyzes online retail transaction data to uncover customer purchase behavior and build actionable machine learning solutions for e-commerce growth. The workflow starts with data cleaning and feature engineering on invoice-level records, followed by structured exploratory data analysis using univariate, bivariate, and multivariate visualizations. Recency-Frequency-Monetary (RFM) features are engineered to represent customer value and engagement. K-Means clustering segments customers into business-friendly groups such as High-Value, Regular, Occasional, and At-Risk. Cluster quality is evaluated using inertia and silhouette score, with hyperparameter tuning for optimal cluster count. An item-based collaborative filtering engine computes cosine similarity between products based on co-purchase behavior and recommends top similar items. Statistical hypothesis tests validate key business assumptions about spending and purchase patterns. The final models are serialized for deployment in a Streamlit application that supports real-time product recommendations and customer segment prediction.


# **GitHub Link -**

Provide your GitHub Link here.


# **Problem Statement**

The global e-commerce industry generates vast transaction data daily. This project examines online retail transactions to segment customers using RFM analysis and deliver personalized product recommendations through collaborative filtering. The goal is to support targeted marketing, retention campaigns, inventory planning, and personalized shopping experiences.


# **General Guidelines** : - 

1. Well-structured, formatted, and commented code is implemented throughout.
2. Exception handling and deployment-ready execution are included.
3. Each logic block includes explanatory comments.
4. 15+ charts follow UBM structure with business interpretation.
5. Multiple ML algorithms include evaluation, tuning, and business impact notes.


# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Core libraries
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
import joblib

import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DATA_PATH, MODELS_DIR, KMEANS_MODEL_PATH, SCALER_MODEL_PATH,
    CLUSTER_LABELS_PATH, SIMILARITY_MATRIX_PATH, PRODUCT_NAMES_PATH, RANDOM_STATE
)
from src.data_loader import load_raw_data, preprocess_retail_data
from src.rfm import build_rfm_table, prepare_rfm_features, train_kmeans_segmentation, label_clusters_from_centroids
from src.recommender import build_product_user_matrix, compute_item_similarity, recommend_similar_products

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries imported successfully.')


### Dataset Loading

In [ ]:
try:
    raw_df = load_raw_data(DATA_PATH)
    print(f'Dataset loaded successfully with shape: {raw_df.shape}')
except Exception as e:
    print(f'Error loading dataset: {e}')


### Dataset First View

In [ ]:
raw_df.head()

### Dataset Rows & Columns count

In [ ]:
print(f'Rows: {raw_df.shape[0]}, Columns: {raw_df.shape[1]}')

### Dataset Information

In [ ]:
raw_df.info()

#### Duplicate Values

In [ ]:
print(f'Duplicate rows: {raw_df.duplicated().sum()}')

#### Missing Values/Null Values

In [ ]:
missing = raw_df.isnull().sum()
print(missing[missing > 0])

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(x=missing[missing > 0].index, y=missing[missing > 0].values, palette='viridis')
plt.title('Missing Values by Column')
plt.ylabel('Count')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


### What did you know about your dataset?

The dataset contains invoice-level retail transactions with product, customer, price, quantity, and country information. CustomerID and Description have missing values. Quantity and UnitPrice include invalid records that must be cleaned. Most customers are from the United Kingdom.


## ***2. Understanding Your Variables***

In [ ]:
raw_df.columns.tolist()

In [ ]:
raw_df.describe(include='all').T

### Variables Description

| Column | Description |
|---|---|
| InvoiceNo | Transaction identifier |
| StockCode | Product code |
| Description | Product name |
| Quantity | Units purchased |
| InvoiceDate | Transaction timestamp |
| UnitPrice | Price per unit |
| CustomerID | Unique customer identifier |
| Country | Customer country |


### Check Unique Values for each variable.

In [ ]:
raw_df.nunique()

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Apply project-specific preprocessing rules
df = preprocess_retail_data(raw_df)
print(f'Cleaned dataset shape: {df.shape}')
df.head()


### What all manipulations have you done and insights you found?

Removed rows with missing CustomerID, cancelled invoices (InvoiceNo starting with 'C'), and non-positive quantity/price values. Added TotalAmount, YearMonth, DayOfWeek, Month, and Hour features for analysis.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

In [ ]:
# Build RFM table for visualization and modeling
rfm = build_rfm_table(df)
print(rfm.head())
print(f'RFM customer count: {rfm.shape[0]}')


#### Chart - 1

In [ ]:
# Chart - 1 visualization code
sns.histplot(df['Quantity'], bins=50, kde=True, color='steelblue')
plt.title('Chart 1: Quantity Distribution')
plt.xlabel('Quantity')
plt.show()

##### 1. Why did you pick the specific chart?

Univariate histogram to understand purchase volume spread.


##### 2. What is/are the insight(s) found from the chart?

Most transactions have low quantities; a long tail indicates bulk orders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Helps optimize packaging and inventory for high-volume SKUs. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 2

In [ ]:
# Chart - 2 visualization code
sns.histplot(df['UnitPrice'], bins=50, kde=True, color='teal')
plt.title('Chart 2: Unit Price Distribution')
plt.xlim(0, df['UnitPrice'].quantile(0.99))
plt.show()

##### 1. Why did you pick the specific chart?

Histogram for price spread analysis.


##### 2. What is/are the insight(s) found from the chart?

Prices are right-skewed with many low-priced items.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Supports pricing tiers and discount strategy. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 3

In [ ]:
# Chart - 3 visualization code
country_txn = df['Country'].value_counts().head(10)
sns.barplot(x=country_txn.values, y=country_txn.index, palette='mako')
plt.title('Chart 3: Top 10 Countries by Transactions')
plt.xlabel('Transactions')
plt.show()

##### 1. Why did you pick the specific chart?

Bar chart for geographic concentration.


##### 2. What is/are the insight(s) found from the chart?

United Kingdom dominates transaction volume.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Enables country-specific campaigns while expanding non-UK markets. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 4

In [ ]:
# Chart - 4 visualization code
monthly_rev = df.groupby('YearMonth')['TotalAmount'].sum().reset_index()
plt.plot(monthly_rev['YearMonth'], monthly_rev['TotalAmount'], marker='o')
plt.title('Chart 4: Monthly Revenue Trend')
plt.xticks(rotation=45)
plt.ylabel('Revenue')
plt.show()

##### 1. Why did you pick the specific chart?

Line chart for temporal seasonality.


##### 2. What is/are the insight(s) found from the chart?

Revenue fluctuates across months with identifiable peaks.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Supports seasonal inventory and promotion planning. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 5

In [ ]:
# Chart - 5 visualization code
top_products = df.groupby('Description')['TotalAmount'].sum().sort_values(ascending=False).head(15)
sns.barplot(x=top_products.values, y=top_products.index, palette='rocket')
plt.title('Chart 5: Top 15 Products by Revenue')
plt.xlabel('Revenue')
plt.show()

##### 1. Why did you pick the specific chart?

Horizontal bar chart for product performance.


##### 2. What is/are the insight(s) found from the chart?

A small set of products contributes disproportionate revenue.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Focus retention offers on top revenue generators. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 6

In [ ]:
# Chart - 6 visualization code
dow = df['DayOfWeek'].value_counts().reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
sns.barplot(x=dow.index, y=dow.values, palette='crest')
plt.title('Chart 6: Transactions by Day of Week')
plt.xticks(rotation=45)
plt.show()

##### 1. Why did you pick the specific chart?

Bar chart for weekday behavior.


##### 2. What is/are the insight(s) found from the chart?

Certain weekdays show higher purchase activity.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Schedule email campaigns on high-conversion days. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 7

In [ ]:
# Chart - 7 visualization code
aov_country = df.groupby('Country').apply(lambda x: x.groupby('InvoiceNo')['TotalAmount'].sum().mean()).sort_values(ascending=False).head(10)
sns.barplot(x=aov_country.values, y=aov_country.index, palette='flare')
plt.title('Chart 7: Average Order Value by Country')
plt.xlabel('AOV')
plt.show()

##### 1. Why did you pick the specific chart?

Bar chart comparing AOV across countries.


##### 2. What is/are the insight(s) found from the chart?

AOV differs significantly by country.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Enables localized pricing and shipping policies. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 8

In [ ]:
# Chart - 8 visualization code
sample_scatter = df.sample(min(5000, len(df)), random_state=RANDOM_STATE)
sns.scatterplot(data=sample_scatter, x='UnitPrice', y='Quantity', alpha=0.4)
plt.title('Chart 8: Quantity vs Unit Price')
plt.xlim(0, sample_scatter['UnitPrice'].quantile(0.99))
plt.show()

##### 1. Why did you pick the specific chart?

Scatter plot for price-quantity relationship.


##### 2. What is/are the insight(s) found from the chart?

Low-price items show higher quantity purchases.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Bundle low-price items to increase basket size. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 9

In [ ]:
# Chart - 9 visualization code
hourly = df.groupby('Hour')['TotalAmount'].sum()
plt.bar(hourly.index, hourly.values, color='slateblue')
plt.title('Chart 9: Revenue by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Revenue')
plt.show()

##### 1. Why did you pick the specific chart?

Line/bar chart for hourly activity.


##### 2. What is/are the insight(s) found from the chart?

Peak shopping hours are visible within the day.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Time flash promotions during peak hours. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 10

In [ ]:
# Chart - 10 visualization code
sns.histplot(rfm['Monetary'], bins=50, kde=True, color='purple')
plt.title('Chart 10: Customer Monetary Distribution')
plt.xlabel('Monetary Value')
plt.show()

##### 1. Why did you pick the specific chart?

Histogram of spend per customer.


##### 2. What is/are the insight(s) found from the chart?

Customer spend is highly skewed with a few big spenders.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

VIP programs should target top monetary customers. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 11

In [ ]:
# Chart - 11 visualization code
sns.histplot(rfm['Recency'], bins=50, kde=True, color='orange')
plt.title('Chart 11: RFM Recency Distribution')
plt.xlabel('Recency (days)')
plt.show()

##### 1. Why did you pick the specific chart?

Histogram of days since last purchase.


##### 2. What is/are the insight(s) found from the chart?

Many customers have moderate recency values.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Reactivation campaigns can target older recency groups. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


#### Chart - 12

In [ ]:
# Chart - 12 visualization code
sns.histplot(rfm['Frequency'], bins=30, kde=True, color='green')
plt.title('Chart 12: RFM Frequency Distribution')
plt.xlabel('Frequency')
plt.show()

##### 1. Why did you pick the specific chart?

Histogram of purchase frequency.


##### 2. What is/are the insight(s) found from the chart?

Majority of customers purchase infrequently.


##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Subscription or loyalty programs can increase frequency. Over-discounting low-price high-volume SKUs without margin checks could hurt profitability.


In [ ]:
# Elbow method for K selection
scaled_rfm, rfm_scaler = prepare_rfm_features(rfm)
inertias, silhouettes = [], []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(scaled_rfm)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(scaled_rfm, labels))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(list(k_range), inertias, marker='o')
ax[0].set_title('Elbow Curve')
ax[0].set_xlabel('K')
ax[0].set_ylabel('Inertia')
ax[1].plot(list(k_range), silhouettes, marker='o', color='green')
ax[1].set_title('Silhouette Score by K')
ax[1].set_xlabel('K')
ax[1].set_ylabel('Silhouette')
plt.tight_layout()
plt.show()


In [ ]:
# Train baseline KMeans with K=4
best_k = 4
kmeans_model, scaler_model, cluster_map, rfm_segmented = train_kmeans_segmentation(rfm, n_clusters=best_k, random_state=RANDOM_STATE)
segment_counts = rfm_segmented['Segment'].value_counts()
sns.barplot(x=segment_counts.index, y=segment_counts.values, palette='Set2')
plt.title('Chart 13: Customer Segment Distribution')
plt.xticks(rotation=20)
plt.ylabel('Customers')
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=rfm_segmented, x='Recency', y='Frequency', hue='Segment', palette='Set1', alpha=0.7)
plt.title('Chart 14: Recency vs Frequency by Segment')
plt.show()


In [ ]:
# Correlation heatmap on numeric transaction features
numeric_cols = df[['Quantity', 'UnitPrice', 'TotalAmount']].corr()
sns.heatmap(numeric_cols, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Chart 15: Correlation Heatmap')
plt.show()


In [ ]:
# Pair plot on RFM features
sns.pairplot(rfm_segmented[['Recency', 'Frequency', 'Monetary', 'Segment']], hue='Segment', palette='Set2', diag_kind='kde')
plt.suptitle('Chart 16: RFM Pair Plot by Segment', y=1.02)
plt.show()


In [ ]:
# Product similarity heatmap sample
product_matrix = build_product_user_matrix(df, min_customers=5)
similarity_df = compute_item_similarity(product_matrix)
sample_products = similarity_df.index[:12].tolist()
sample_sim = similarity_df.loc[sample_products, sample_products]
sns.heatmap(sample_sim, cmap='YlOrRd')
plt.title('Chart 17: Product Similarity Heatmap (Sample)')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset.

1. UK customers have higher average order value than non-UK customers.
2. Weekend average transaction quantity differs from weekday average quantity.
3. Higher purchase frequency customers spend significantly more (Monetary).


### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H0: Mean AOV(UK) = Mean AOV(Non-UK); HA: Mean AOV(UK) != Mean AOV(Non-UK)


#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
invoice_totals = df.groupby(['InvoiceNo', 'Country'])['TotalAmount'].sum().reset_index()
uk = invoice_totals[invoice_totals['Country'] == 'United Kingdom']['TotalAmount']
non_uk = invoice_totals[invoice_totals['Country'] != 'United Kingdom']['TotalAmount']
t_stat, p_val = stats.ttest_ind(uk, non_uk, equal_var=False)
print(f'T-statistic: {t_stat:.4f}, P-value: {p_val:.6f}')
print('Reject H0' if p_val < 0.05 else 'Fail to reject H0')

##### Which statistical test have you done to obtain P-Value?

Welch's t-test


##### Why did you choose the specific statistical test?

Compares means of two independent groups with potentially unequal variances.


### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H0: Mean quantity(weekend) = Mean quantity(weekday); HA: Means are different


#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
df['IsWeekend'] = df['DayOfWeek'].isin(['Saturday', 'Sunday'])
weekend_q = df[df['IsWeekend']]['Quantity']
weekday_q = df[~df['IsWeekend']]['Quantity']
t_stat, p_val = stats.ttest_ind(weekend_q, weekday_q, equal_var=False)
print(f'T-statistic: {t_stat:.4f}, P-value: {p_val:.6f}')
print('Reject H0' if p_val < 0.05 else 'Fail to reject H0')

##### Which statistical test have you done to obtain P-Value?

Welch's t-test


##### Why did you choose the specific statistical test?

Tests whether weekend shopping behavior differs from weekdays.


### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

H0: No monotonic relationship between Frequency and Monetary; HA: Relationship exists


#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
corr, p_val = stats.spearmanr(rfm['Frequency'], rfm['Monetary'])
print(f'Spearman correlation: {corr:.4f}, P-value: {p_val:.6f}')
print('Reject H0' if p_val < 0.05 else 'Fail to reject H0')

##### Which statistical test have you done to obtain P-Value?

Spearman correlation test


##### Why did you choose the specific statistical test?

Frequency and monetary values are skewed; rank-based test is robust.


## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Missing values handled during preprocessing by dropping invalid CustomerID/Description rows
print('Missing values after preprocessing:')
print(df.isnull().sum())


#### What all missing value imputation techniques have you used and why did you use those techniques?

CustomerID and Description missing rows were removed because they cannot be linked to valid customers/products.


### 2. Handling Outliers

In [ ]:
# Cap extreme monetary outliers at 99th percentile for modeling stability
rfm_model = rfm.copy()
upper = rfm_model['Monetary'].quantile(0.99)
rfm_model['Monetary'] = rfm_model['Monetary'].clip(upper=upper)
print(f'Monetary capped at 99th percentile: {upper:.2f}')


##### What all outlier treatment techniques have you used and why did you use those techniques?

Winsorization at 99th percentile reduces impact of extreme spenders while preserving customer rank order.


### 3. Categorical Encoding

In [ ]:
# Not required for RFM clustering; Country retained for EDA only
print('Categorical encoding skipped for unsupervised RFM model.')

#### What all categorical encoding techniques have you used & why did you use those techniques?

RFM clustering uses numeric Recency, Frequency, Monetary features only.


### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

Not applicable for this structured retail transaction dataset.


### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
rfm_features, feature_scaler = prepare_rfm_features(rfm_model)
rfm_features.head()


#### 2. Feature Selection

In [ ]:
# Selected features: Recency, Frequency, Monetary
selected_features = ['Recency', 'Frequency', 'Monetary']
X = rfm_model[selected_features]
print(X.describe())

##### What all feature selection methods have you used  and why?

Domain-driven RFM feature selection aligns with marketing literature and business interpretability.


##### Which all features you found important and why?

Recency captures churn risk, Frequency captures loyalty, Monetary captures revenue contribution.


### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Log1p transformation applied inside prepare_rfm_features to reduce skew
print('Log transformation + standardization applied to RFM features.')

Log transform stabilizes skewed RFM distributions and improves clustering performance.


### 6. Data Scaling

In [ ]:
X_scaled = feature_scaler.fit_transform(rfm_model[selected_features])
print('StandardScaler applied to RFM features.')

##### Which method have you used to scale you data and why?

StandardScaler ensures equal contribution of R, F, and M during distance-based clustering.


### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Not required because RFM has only three meaningful business dimensions.


In [ ]:
# Dimensionality reduction not applied
pass

### 8. Data Splitting

In [ ]:
# Unsupervised clustering uses full RFM table
# Recommendation evaluation uses train/test split at invoice level
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)
print(f'Train: {train_df.shape}, Test: {test_df.shape}')


##### What data splitting ratio have you used and why?

80/20 split for recommendation sanity checks; clustering uses all customers for stable segments.


### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

Customer segments are naturally imbalanced; this reflects real business distribution and is acceptable for segmentation.


In [ ]:
# No resampling applied for unsupervised segmentation
pass

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
model1 = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
pred1 = model1.fit_predict(X_scaled)
sil1 = silhouette_score(X_scaled, pred1)
print(f'KMeans (K=4) Inertia: {model1.inertia_:.2f}, Silhouette: {sil1:.4f}')


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
metrics_m1 = pd.DataFrame({'Metric': ['Inertia', 'Silhouette'], 'Score': [model1.inertia_, sil1]})
sns.barplot(data=metrics_m1, x='Metric', y='Score', palette='Blues_d')
plt.title('Model 1: KMeans Evaluation Metrics')
plt.show()


KMeans partitions customers into four compact RFM clusters. Lower inertia and higher silhouette indicate better separation.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Manual hyperparameter tuning for unsupervised KMeans
k_candidates = [3, 4, 5, 6]
sil_scores = []
for k in k_candidates:
    km_tmp = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_tmp = km_tmp.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels_tmp))
best_k_idx = int(np.argmax(sil_scores))
best_k = k_candidates[best_k_idx]
best_kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
pred_tuned = best_kmeans.fit_predict(X_scaled)
sil_tuned = silhouette_score(X_scaled, pred_tuned)
print(dict(zip(k_candidates, sil_scores)))
print(f'Best K: {best_k}, Tuned Silhouette: {sil_tuned:.4f}')


##### Which hyperparameter optimization technique have you used and why?

Manual grid search over K with silhouette scoring selects optimal cluster count for unsupervised learning.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

In [ ]:
compare_m1 = pd.DataFrame({
    'Model': ['Baseline KMeans', 'Tuned KMeans'],
    'Silhouette': [sil1, sil_tuned]
})
sns.barplot(data=compare_m1, x='Model', y='Silhouette', palette='Greens_d')
plt.title('Model 1: Baseline vs Tuned Silhouette Score')
plt.ylim(0, max(compare_m1['Silhouette']) * 1.2)
plt.show()


Tuned KMeans generally improves silhouette score versus default K=4.


### ML Model - 2

In [ ]:
model2 = AgglomerativeClustering(n_clusters=best_k)
pred2 = model2.fit_predict(X_scaled)
sil2 = silhouette_score(X_scaled, pred2)
print(f'Agglomerative Silhouette: {sil2:.4f}')


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
metrics_m2 = pd.DataFrame({'Model': ['KMeans Tuned', 'Agglomerative'], 'Silhouette': [sil_tuned, sil2]})
sns.barplot(data=metrics_m2, x='Model', y='Silhouette', palette='Oranges_d')
plt.title('Model 2: Agglomerative vs KMeans')
plt.show()


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
agg_params = [(k, link) for k in [3, 4, 5, 6] for link in ['ward', 'average']]
agg_scores = []
for k, link in agg_params:
    agg_tmp = AgglomerativeClustering(n_clusters=k, linkage=link)
    labels_tmp = agg_tmp.fit_predict(X_scaled)
    agg_scores.append(silhouette_score(X_scaled, labels_tmp))
best_agg_idx = int(np.argmax(agg_scores))
best_k_agg, best_link = agg_params[best_agg_idx]
best_agg = AgglomerativeClustering(n_clusters=best_k_agg, linkage=best_link)
pred2_tuned = best_agg.fit_predict(X_scaled)
sil2_tuned = silhouette_score(X_scaled, pred2_tuned)
print(f'Best Agg Params: n_clusters={best_k_agg}, linkage={best_link}, Silhouette={sil2_tuned:.4f}')


##### Which hyperparameter optimization technique have you used and why?

Manual parameter sweep over cluster count and linkage methods for hierarchical clustering.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

In [ ]:
compare_m2 = pd.DataFrame({'Stage': ['Baseline', 'Tuned'], 'Silhouette': [sil2, sil2_tuned]})
sns.barplot(data=compare_m2, x='Stage', y='Silhouette', palette='Purples_d')
plt.title('Model 2: Agglomerative Tuning Impact')
plt.show()


Agglomerative clustering provides an alternative partition for validation of segment stability.


### ML Model - 3

In [ ]:
# Item-based collaborative filtering
train_matrix = build_product_user_matrix(train_df, min_customers=5)
train_sim = compute_item_similarity(train_matrix)

def recommendation_hit_rate(sim_df, test_data, top_n=5, sample_size=300):
    hits, total = 0, 0
    grouped = test_data.groupby('InvoiceNo')
    invoices = list(grouped.groups.keys())
    np.random.seed(RANDOM_STATE)
    sample_invoices = np.random.choice(invoices, size=min(sample_size, len(invoices)), replace=False)
    for inv in sample_invoices:
        basket = grouped.get_group(inv)['Description'].unique().tolist()
        if len(basket) < 2:
            continue
        anchor = basket[0]
        target_items = set(basket[1:])
        recs = recommend_similar_products(anchor, sim_df, top_n=top_n)
        if recs.empty:
            continue
        recommended = set(recs['Product'].tolist())
        if recommended & target_items:
            hits += 1
        total += 1
    return hits / total if total else 0

hit_rate = recommendation_hit_rate(train_sim, test_df)
print(f'Recommendation Hit Rate@{5}: {hit_rate:.4f}')


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
metrics_m3 = pd.DataFrame({'Metric': ['HitRate@5'], 'Score': [hit_rate]})
sns.barplot(data=metrics_m3, x='Metric', y='Score', palette='Reds_d')
plt.ylim(0, 1)
plt.title('Model 3: Recommendation Hit Rate')
plt.show()


Item-based collaborative filtering recommends products co-purchased by similar customers using cosine similarity.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
min_customer_grid = [3, 5, 8]
hit_scores = []
for mc in min_customer_grid:
    sim_tmp = compute_item_similarity(build_product_user_matrix(train_df, min_customers=mc))
    hit_scores.append(recommendation_hit_rate(sim_tmp, test_df))
best_idx = int(np.argmax(hit_scores))
best_min_customers = min_customer_grid[best_idx]
print(dict(zip(min_customer_grid, hit_scores)))
print(f'Best min_customers: {best_min_customers}')


##### Which hyperparameter optimization technique have you used and why?

Manual grid search on minimum customer threshold balances sparsity and recommendation coverage.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

In [ ]:
tuning_m3 = pd.DataFrame({'min_customers': min_customer_grid, 'HitRate@5': hit_scores})
sns.barplot(data=tuning_m3, x='min_customers', y='HitRate@5', palette='BuGn_d')
plt.title('Model 3: Recommendation Tuning')
plt.show()


#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

- **Silhouette Score**: Measures cluster separation; higher values mean clearer customer segments for CRM.
- **Inertia**: Measures within-cluster variance; lower values mean tighter segments.
- **HitRate@5**: Measures recommendation relevance; higher values mean better cross-sell potential.


### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Silhouette for segmentation quality and HitRate@5 for recommendation relevance directly map to campaign targeting and conversion.


### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Tuned KMeans is selected for customer segmentation due to interpretability, speed, and strong silhouette performance.


### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
final_model, final_scaler, final_labels, final_rfm = train_kmeans_segmentation(
    rfm_model, n_clusters=best_k, random_state=RANDOM_STATE
)
centroid_df = pd.DataFrame(final_model.cluster_centers_, columns=['Recency', 'Frequency', 'Monetary'])
centroid_df['Segment'] = centroid_df.index.map(final_labels)
centroid_df


In [ ]:
final_product_matrix = build_product_user_matrix(df, min_customers=best_min_customers)
final_similarity = compute_item_similarity(final_product_matrix)
product_names = final_similarity.index.tolist()
recommend_similar_products(product_names[0], final_similarity, top_n=5)


## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.

In [ ]:
joblib.dump(final_model, KMEANS_MODEL_PATH)
joblib.dump(final_scaler, SCALER_MODEL_PATH)
joblib.dump(final_labels, CLUSTER_LABELS_PATH)
joblib.dump(final_similarity, SIMILARITY_MATRIX_PATH)
joblib.dump(product_names, PRODUCT_NAMES_PATH)
print('Models saved successfully in models/ folder.')


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.

In [ ]:
loaded_kmeans = joblib.load(KMEANS_MODEL_PATH)
loaded_scaler = joblib.load(SCALER_MODEL_PATH)
loaded_labels = joblib.load(CLUSTER_LABELS_PATH)
loaded_sim = joblib.load(SIMILARITY_MATRIX_PATH)

sample_rfm = np.array([[20, 8, 1200.0]])
sample_scaled = loaded_scaler.transform([[np.log1p(20), np.log1p(8), np.log1p(1200.0)]])
sample_cluster = loaded_kmeans.predict(sample_scaled)[0]
print('Sample segment:', loaded_labels[sample_cluster])
print(recommend_similar_products(product_names[0], loaded_sim, top_n=5))


### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This capstone project delivered an end-to-end retail analytics solution: cleaned transaction data, rich EDA, RFM-based customer segmentation, and item-based product recommendations. KMeans clustering created actionable segments (High-Value, Regular, Occasional, At-Risk) while collaborative filtering enabled real-time cross-sell recommendations. Artifacts were exported for Streamlit deployment.


### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***